# AnyUp 2D vs AnyUp 3D — Temporal Consistency Comparison

**Feature extractor:** VideoMAE (`MCG-NJU/videomae-base`) — the same frozen teacher used  
to supervise AnyUp3D. Both models receive **identical** VideoMAE feature inputs from the same video.

| Model | What it processes |
|-------|-------------------|
| **AnyUp 2D** (`anyup/anyup/model_2d.py :: AnyUp2D`) | `for t in range(T_EFF)` — one feature map at a time, no temporal context |
| **AnyUp 3D** (`anyup/model.py :: AnyUp`) | single forward on the full `(B, C, T_EFF, h, w)` volume |

Both models receive guidance images at **the same `TARGET_SIZE × TARGET_SIZE` resolution**.

## Cell 1 — Install & mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Must be set before any CUDA allocation
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!pip install -q transformers einops av

# Clone the repo (branch 3Dconv has both AnyUp2D and AnyUp3D)
!git clone -b 3Dconv https://github.com/mu-az88/anyup /content/anyup 2>/dev/null || echo 'already cloned'
!pip install -q -e /content/anyup

import torch
print('CUDA:', torch.cuda.is_available())
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## Cell 2 — Config  ← THE ONLY CELL YOU NEED TO EDIT

Change paths and knobs here. Everything downstream derives from these values.

In [ ]:
# ── PATHS ─────────────────────────────────────────────────────────────────────
CHECKPOINT_DIR = '/content/drive/MyDrive/anyup3d/checkpoints/run_01'  # AnyUp3D checkpoint folder
ANYUP2D_CKPT   = '/content/anyup/anyup_2d.pth'                        # 2D weights (already in repo)
VIDEO_DIR      = '/content/drive/MyDrive/anyup3d/test_videos'          # drop your .mp4 files here

# ── MODEL KNOBS — must match what you used during training ────────────────────
TARGET_SIZE = 112    # ↓ H=W fed to VideoMAE and BOTH AnyUp models (same resolution, always)
                     #   112 fits comfortably on T4; set to 224 if you have memory headroom
T_INPUT     = 16     # ↓ raw frames fed to VideoMAE; must be even (tubelet_size=2)
                     #   16 is VideoMAE's native length — no pos-embed interpolation needed
                     #   do NOT set below 4

# ── EVAL KNOBS ────────────────────────────────────────────────────────────────
NUM_TEST_CLIPS   = 2      # ↓ number of video clips to evaluate; reduce for a quick check
STATIC_THRESHOLD = 0.02   # RGB-diff gate for temporal variance: pairs where the scene
                           #   changed more than this are excluded from the flicker metric

# ── VIS KNOBS ─────────────────────────────────────────────────────────────────
VIZ_FRAMES = 4   # ↓ number of temporal tokens to show in PCA grid (max = T_EFF)

# ── VideoMAE constants — FIXED by the pretrained model, do not change ─────────
ENCODER_MODEL  = 'MCG-NJU/videomae-base'
TUBELET_SIZE   = 2    # two consecutive raw frames per temporal token
PATCH_SIZE     = 16   # spatial patch size
FEAT_DIM       = 768  # VideoMAE-Base hidden dim

# ── Derived — do not edit ──────────────────────────────────────────────────────
import torch, sys, numpy as np, matplotlib.pyplot as plt, torch.nn.functional as F
from pathlib import Path
from sklearn.decomposition import PCA
sys.path.insert(0, '/content/anyup')

T_EFF   = T_INPUT // TUBELET_SIZE     # temporal feature tokens, e.g. 16//2 = 8
h_tok   = TARGET_SIZE // PATCH_SIZE   # spatial token grid height, e.g. 112//16 = 7
w_tok   = TARGET_SIZE // PATCH_SIZE
N_TOK   = T_EFF * h_tok * w_tok       # total flat tokens, e.g. 8*7*7 = 392
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print(f'Video  : {T_INPUT} raw frames  →  {T_EFF} temporal tokens')
print(f'Spatial: {TARGET_SIZE}x{TARGET_SIZE}  →  {h_tok}x{w_tok} token grid')
print(f'Tokens : {N_TOK} total per clip   feat_dim={FEAT_DIM}')
print(f'Device : {DEVICE}')

## Cell 3 — Load VideoMAE (shared frozen feature extractor)

In [ ]:
from transformers import VideoMAEModel

videomae = VideoMAEModel.from_pretrained(ENCODER_MODEL)
videomae = videomae.to(DEVICE).eval()
for p in videomae.parameters():
    p.requires_grad_(False)

# Verify config constants match the actual model
assert videomae.config.tubelet_size == TUBELET_SIZE, 'TUBELET_SIZE mismatch'
assert videomae.config.patch_size   == PATCH_SIZE,   'PATCH_SIZE mismatch'
assert videomae.config.hidden_size  == FEAT_DIM,     'FEAT_DIM mismatch'
print(f'VideoMAE loaded: {ENCODER_MODEL}')

# ImageNet normalisation tensors — shape broadcast-compatible with (B, T, C, H, W)
_MEAN = torch.tensor(IMAGENET_MEAN, device=DEVICE).view(1,1,3,1,1)
_STD  = torch.tensor(IMAGENET_STD,  device=DEVICE).view(1,1,3,1,1)


@torch.no_grad()
def extract_videomae_features(video_BTCHW: torch.Tensor) -> torch.Tensor:
    """
    Run VideoMAE on one video clip and return the feature volume.

    Args:
        video_BTCHW : (1, T_INPUT, 3, TARGET_SIZE, TARGET_SIZE)  float32, values in [0,1]

    Returns:
        feat_vol : (1, FEAT_DIM, T_EFF, h_tok, w_tok)  channels-first, on GPU
                   This is the SHARED input for both AnyUp2D (sliced per token)
                   and AnyUp3D (full volume).
    """
    B, T, C, H, W = video_BTCHW.shape
    assert T == T_INPUT, f'Need T_INPUT={T_INPUT} frames, got {T}'
    assert H == TARGET_SIZE and W == TARGET_SIZE, \
        f'Resolution must be {TARGET_SIZE}x{TARGET_SIZE}, got {H}x{W}'

    x      = (video_BTCHW.to(DEVICE) - _MEAN) / _STD   # ImageNet normalise
    tokens = videomae(pixel_values=x).last_hidden_state  # (B, N_TOK, FEAT_DIM)

    assert tokens.shape[1] == N_TOK, \
        f'Token count mismatch: expected {N_TOK}, got {tokens.shape[1]}'

    # (B, N_TOK, C) → (B, T_EFF, h_tok, w_tok, C) → (B, C, T_EFF, h_tok, w_tok)
    feat_vol = tokens.reshape(B, T_EFF, h_tok, w_tok, FEAT_DIM)
    feat_vol = feat_vol.permute(0, 4, 1, 2, 3).contiguous()
    return feat_vol   # (1, FEAT_DIM, T_EFF, h_tok, w_tok)


# Sanity check
_t = extract_videomae_features(torch.rand(1, T_INPUT, 3, TARGET_SIZE, TARGET_SIZE))
print(f'VideoMAE output: {_t.shape}  ✓' if _t.shape == (1, FEAT_DIM, T_EFF, h_tok, w_tok)
      else f'Shape mismatch: {_t.shape}')
del _t; torch.cuda.empty_cache()

## Cell 4 — Load AnyUp 2D (unmodified original model)

In [ ]:
# AnyUp2D lives at anyup/anyup/model_2d.py in the repo.
# Loaded from the local checkpoint — zero modifications to the class.
#
# Official call signature (from the AnyUp README, channels-FIRST):
#   hr_features = anyup2d(hr_image, lr_features)
#   hr_image    : (B, 3, H, W)   ImageNet-normalised, channels-first
#   lr_features : (B, C, h, w)   channels-first
#   → returns   : (B, C, H, W)   channels-first upsampled features
#
# H = W = TARGET_SIZE  (same resolution used for AnyUp3D)

from anyup.anyup.model_2d import AnyUp2D   # adjust path if import fails

anyup2d = AnyUp2D()
state_2d = torch.load(ANYUP2D_CKPT, map_location='cpu')
# The checkpoint may or may not have a wrapper key
state_2d = state_2d.get('model_state_dict', state_2d.get('state_dict', state_2d))
anyup2d.load_state_dict(state_2d, strict=True)
anyup2d = anyup2d.to(DEVICE).eval()

print(f'AnyUp 2D loaded from {ANYUP2D_CKPT}')
print(f'  params: {sum(p.numel() for p in anyup2d.parameters())/1e6:.1f}M')

# Pre-build ImageNet normalisation for AnyUp2D guidance images
# (separate from the VideoMAE normalisation tensors above)
_MEAN_2D = torch.tensor(IMAGENET_MEAN, device=DEVICE).view(1,3,1,1)  # (1,3,1,1) for (B,C,H,W)
_STD_2D  = torch.tensor(IMAGENET_STD,  device=DEVICE).view(1,3,1,1)

## Cell 5 — Load AnyUp 3D

In [ ]:
# AnyUp3D lives at anyup/model.py, class AnyUp.
#
# Call signature (channels-first 5D):
#   hr_features = anyup3d(hr_image_vol, lr_feat_vol)
#   hr_image_vol : (B, 3, T_EFF, TARGET_SIZE, TARGET_SIZE)  ImageNet-normalised
#   lr_feat_vol  : (B, FEAT_DIM, T_EFF, h_tok, w_tok)
#   → returns    : (B, FEAT_DIM, T_EFF, TARGET_SIZE, TARGET_SIZE)

from anyup.model import AnyUp as AnyUp3D   # adjust if class name differs

anyup3d = AnyUp3D().to(DEVICE).eval()

# Find the best checkpoint in the checkpoint directory
ckpt_dir = Path(CHECKPOINT_DIR)
ckpt_files = sorted(ckpt_dir.glob('*.pth'))
assert ckpt_files, f'No .pth files found in {CHECKPOINT_DIR}'
# Prefer 'best.pth'; fall back to the last file by name sort
best = next((f for f in ckpt_files if 'best' in f.name), ckpt_files[-1])
print(f'Loading AnyUp3D checkpoint: {best}')

ckpt = torch.load(best, map_location='cpu')
state_3d = ckpt.get('model_state_dict', ckpt.get('state_dict', ckpt))
anyup3d.load_state_dict(state_3d, strict=True)

print(f'AnyUp 3D loaded  — step {ckpt.get("global_step", "?")}')
print(f'  params: {sum(p.numel() for p in anyup3d.parameters())/1e6:.1f}M')

## Cell 6 — Load test video clips from VIDEO_DIR

In [ ]:
import torchvision
from torchvision import transforms

def load_video_clip(path: Path) -> torch.Tensor:
    """
    Load a video file, take the first T_INPUT frames, resize to TARGET_SIZE.

    Returns:
        (1, T_INPUT, 3, TARGET_SIZE, TARGET_SIZE)  float32 in [0, 1], NOT normalised.
        Normalisation is done inside extract_videomae_features() and the AnyUp calls.
    """
    vframes, _, _ = torchvision.io.read_video(str(path), pts_unit='sec', output_format='TCHW')
    # vframes: (T_full, 3, H_orig, W_orig)  uint8

    T_full = vframes.shape[0]
    if T_full < T_INPUT:
        # Loop short clips to reach T_INPUT frames
        reps   = math.ceil(T_INPUT / T_full)
        vframes = vframes.repeat(reps, 1, 1, 1)
    vframes = vframes[:T_INPUT]   # (T_INPUT, 3, H, W)

    # Resize to TARGET_SIZE x TARGET_SIZE
    vframes = vframes.float() / 255.0                                 # [0,1]
    vframes = F.interpolate(vframes,
                            size=(TARGET_SIZE, TARGET_SIZE),           # ↑ TARGET_SIZE from Cell 2
                            mode='bilinear', align_corners=False)      # (T_INPUT, 3, S, S)
    return vframes.unsqueeze(0)   # (1, T_INPUT, 3, TARGET_SIZE, TARGET_SIZE)


import math
video_dir = Path(VIDEO_DIR)
video_paths = sorted(video_dir.glob('*.mp4'))[:NUM_TEST_CLIPS]   # ↑ NUM_TEST_CLIPS from Cell 2

if not video_paths:
    print(f'No .mp4 files found in {VIDEO_DIR}.')
    print('Falling back to synthetic clips for smoke-testing.')
    torch.manual_seed(42)
    # Smooth random video: base frame + small per-frame noise
    base = torch.rand(1, 1, 3, TARGET_SIZE, TARGET_SIZE)
    synthetic = torch.cat(
        [(base + torch.randn_like(base)*0.03).clamp(0,1) for _ in range(T_INPUT)], dim=1
    )
    clips = [synthetic]   # list of (1, T_INPUT, 3, S, S)
    clip_names = ['synthetic_clip']
else:
    clips      = [load_video_clip(p) for p in video_paths]
    clip_names = [p.stem for p in video_paths]

print(f'Loaded {len(clips)} clip(s):')
for name, clip in zip(clip_names, clips):
    print(f'  {name}: {clip.shape}  dtype={clip.dtype}')

# Quick visual check — first clip, first 4 frames
n_show = min(4, T_INPUT)
fig, axes = plt.subplots(1, n_show, figsize=(3*n_show, 3))
for i, ax in enumerate(axes):
    ax.imshow(clips[0][0, i].permute(1,2,0).numpy())
    ax.set_title(f'frame {i}'); ax.axis('off')
plt.suptitle(f'{clip_names[0]}  ({T_INPUT} frames, showing {n_show})')
plt.tight_layout(); plt.show()

## Cell 7 — Run all three methods on each clip

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Guidance frame selection:
#   Each temporal token covers TUBELET_SIZE=2 consecutive raw frames.
#   We use the midpoint raw frame of each tubelet as the RGB guidance image.
#   This gives T_EFF guidance frames, one per token, at TARGET_SIZE resolution.
#
# Resolution guarantee:
#   Both AnyUp2D and AnyUp3D receive guidance images at exactly TARGET_SIZE×TARGET_SIZE.
#   The feature maps (h_tok×w_tok) are identical for both models.
# ─────────────────────────────────────────────────────────────────────────────

guidance_idx = [t * TUBELET_SIZE + TUBELET_SIZE // 2 for t in range(T_EFF)]
# e.g. T_EFF=8, TUBELET_SIZE=2 → [1, 3, 5, 7, 9, 11, 13, 15]
print(f'Guidance raw-frame indices: {guidance_idx}')

all_results = {}   # clip_name → {'2d': vol, '3d': vol, 'bilinear': vol, 'raw': clip}

for clip_name, video in zip(clip_names, clips):
    print(f'\n═══ {clip_name} ═══')

    # ── Step A: VideoMAE features (run ONCE, shared by both models) ────────────
    print('  A) VideoMAE...')
    feat_vol = extract_videomae_features(video)
    # feat_vol: (1, FEAT_DIM, T_EFF, h_tok, w_tok)  on GPU

    # ── Build guidance volume: (1, 3, T_EFF, TARGET_SIZE, TARGET_SIZE) ────────
    # video: (1, T_INPUT, 3, S, S) in [0,1] — index T_INPUT dim to pick T_EFF frames
    guidance_raw = video[:, guidance_idx, :, :, :]    # (1, T_EFF, 3, S, S)  in [0,1]
    guidance_raw = guidance_raw.permute(0, 2, 1, 3, 4)  # (1, 3, T_EFF, S, S)

    # Normalise guidance to ImageNet stats for both models
    _m = torch.tensor(IMAGENET_MEAN, device=DEVICE).view(1,3,1,1,1)  # (1,3,1,1,1)
    _s = torch.tensor(IMAGENET_STD,  device=DEVICE).view(1,3,1,1,1)
    guidance_norm = (guidance_raw.to(DEVICE) - _m) / _s   # (1, 3, T_EFF, S, S)

    # ── Step B: AnyUp 2D — one temporal token at a time ───────────────────────
    # Signature: anyup2d(hr_image, lr_features)
    #   hr_image    : (1, 3, TARGET_SIZE, TARGET_SIZE)  ImageNet-normalised, channels-first
    #   lr_features : (1, FEAT_DIM, h_tok, w_tok)       channels-first
    #   → returns   : (1, FEAT_DIM, TARGET_SIZE, TARGET_SIZE)
    print(f'  B) AnyUp 2D ({T_EFF} tokens independently)...')
    out2d_list = []
    with torch.no_grad():
        for t in range(T_EFF):   # ← the defining loop: zero temporal context
            p_t     = feat_vol[:, :, t, :, :]        # (1, FEAT_DIM, h_tok, w_tok)
            I_hr_t  = guidance_norm[:, :, t, :, :]   # (1, 3, TARGET_SIZE, TARGET_SIZE)
            out_t   = anyup2d(I_hr_t, p_t)           # (1, FEAT_DIM, TARGET_SIZE, TARGET_SIZE)
            out2d_list.append(out_t.cpu())
    # Stack: list of (1, C, S, S) → (T_EFF, C, S, S) → (T_EFF, S, S, C) for metrics
    anyup2d_vol = torch.cat(out2d_list, dim=0)           # (T_EFF, FEAT_DIM, S, S)
    anyup2d_vol = anyup2d_vol.permute(0, 2, 3, 1)        # (T_EFF, S, S, FEAT_DIM) channels-last
    print(f'     output: {anyup2d_vol.shape}')

    # ── Step C: AnyUp 3D — all T_EFF tokens jointly ───────────────────────────
    # Signature: anyup3d(hr_image_vol, lr_feat_vol)
    #   hr_image_vol : (1, 3, T_EFF, TARGET_SIZE, TARGET_SIZE)
    #   lr_feat_vol  : (1, FEAT_DIM, T_EFF, h_tok, w_tok)
    #   → returns    : (1, FEAT_DIM, T_EFF, TARGET_SIZE, TARGET_SIZE)
    print(f'  C) AnyUp 3D (all {T_EFF} tokens jointly)...')
    with torch.no_grad():
        out3d = anyup3d(guidance_norm, feat_vol)   # (1, FEAT_DIM, T_EFF, S, S)
    anyup3d_vol = out3d.squeeze(0)                 # (FEAT_DIM, T_EFF, S, S)
    anyup3d_vol = anyup3d_vol.permute(1, 2, 3, 0) # (T_EFF, S, S, FEAT_DIM) channels-last
    anyup3d_vol = anyup3d_vol.cpu().contiguous()
    print(f'     output: {anyup3d_vol.shape}')

    # ── Step D: Bilinear baseline ──────────────────────────────────────────────
    print('  D) Bilinear baseline...')
    bl_list = []
    for t in range(T_EFF):
        p_t = feat_vol[:, :, t, :, :].cpu()             # (1, FEAT_DIM, h_tok, w_tok)
        up  = F.interpolate(p_t, size=(TARGET_SIZE, TARGET_SIZE),
                            mode='bilinear', align_corners=False)  # (1, C, S, S)
        bl_list.append(up.permute(0, 2, 3, 1))                     # (1, S, S, C)
    bilinear_vol = torch.cat(bl_list, dim=0)   # (T_EFF, S, S, FEAT_DIM)

    all_results[clip_name] = {
        'raw'      : video,
        'bilinear' : bilinear_vol,
        '2d'       : anyup2d_vol,
        '3d'       : anyup3d_vol,
    }
    torch.cuda.empty_cache()
    print(f'  Done.')

print('\nAll clips processed.')

## Cell 8 — Compute temporal consistency metrics

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Metric 1 — Temporal Feature Variance
#   For each spatial position, measure the variance of the feature vector
#   across T_EFF tokens. Average over all positions.
#   Lower = features are more stable over time = better temporal consistency.
#
# Metric 2 — Temporal Cosine Drift
#   Mean cosine DISTANCE between consecutive token pairs (t, t+1).
#   Optionally gated: exclude pairs where the scene changed significantly
#   (mean RGB diff > STATIC_THRESHOLD) to focus on flicker in static regions.
#   Lower = smoother feature evolution.
# ─────────────────────────────────────────────────────────────────────────────

def temporal_feature_variance(vol: torch.Tensor) -> float:
    """vol: (T, H, W, C). Returns mean per-position variance across T."""
    return vol.var(dim=0).mean().item()


def temporal_cosine_drift(
    vol:       torch.Tensor,
    raw_video: torch.Tensor = None,   # (1, T_INPUT, 3, H, W) for RGB gate; None = no gate
) -> float:
    """vol: (T_EFF, H, W, C). Returns mean cosine distance between consecutive tokens."""
    T, H, W, C = vol.shape
    flat = vol.reshape(T, H * W, C)
    total, count = 0.0, 0

    for t in range(T - 1):
        # Optional RGB gate: skip pairs with too much scene motion
        if raw_video is not None:
            f0_idx, f1_idx = guidance_idx[t], guidance_idx[t+1]
            rgb_diff = (raw_video[0, f1_idx].float() - raw_video[0, f0_idx].float()).abs().mean().item()
            if rgb_diff > STATIC_THRESHOLD:   # ↑ STATIC_THRESHOLD from Cell 2
                continue   # exclude high-motion pairs from the flicker metric

        sim    = F.cosine_similarity(flat[t], flat[t+1], dim=-1)  # (H*W,)
        total += (1.0 - sim).mean().item()
        count += 1

    return total / count if count > 0 else float('nan')


# ── Compute metrics for every clip and every method ────────────────────────────
all_metrics = {}
for clip_name, res in all_results.items():
    raw = res['raw']
    all_metrics[clip_name] = {}
    for method in ['bilinear', '2d', '3d']:
        vol = res[method]
        all_metrics[clip_name][method] = {
            'var'  : temporal_feature_variance(vol),
            'drift': temporal_cosine_drift(vol, raw),
        }

# ── Print summary table ────────────────────────────────────────────────────────
METHOD_LABELS = {'bilinear':'Bilinear (floor)', '2d':'AnyUp 2D', '3d':'AnyUp 3D'}
print(f'\n{"─"*70}')
for clip_name, metrics in all_metrics.items():
    print(f'Clip: {clip_name}')
    print(f'  {"Method":<22}  {"Feat Variance":>14}  {"Cosine Drift":>14}')
    print(f'  {"─"*54}')
    for m, vals in metrics.items():
        tag = '  ← sanity floor' if m=='bilinear' else ('  ← target' if m=='3d' else '')
        print(f'  {METHOD_LABELS[m]:<22}  {vals["var"]:>14.6f}  {vals["drift"]:>14.6f}{tag}')

    v2, v3 = metrics['2d']['var'],   metrics['3d']['var']
    d2, d3 = metrics['2d']['drift'], metrics['3d']['drift']
    print(f'  Δ (3D − 2D) variance : {v3-v2:+.6f}  (negative = 3D less flickery)')
    print(f'  Δ (3D − 2D) drift    : {d3-d2:+.6f}  (negative = 3D smoother)')
print(f'{"─"*70}')
print('Expected order for a working AnyUp3D:  Bilinear ≤ AnyUp3D < AnyUp2D')

## Cell 9 — PCA visualisation (4-row side-by-side per clip)

In [ ]:
# Shared PCA basis across all three volumes → colours directly comparable.

def fit_shared_pca(*vols, n_components=3, max_pixels=30_000):
    all_feats = []
    for vol in vols:
        flat = vol.reshape(-1, vol.shape[-1]).numpy()
        idx  = np.random.choice(len(flat), min(max_pixels//len(vols), len(flat)), replace=False)
        all_feats.append(flat[idx])
    pca = PCA(n_components=n_components)
    pca.fit(np.concatenate(all_feats, axis=0))
    return pca

def vol_to_pca_rgb(vol, pca):
    """(T, H, W, C) → (T, H, W, 3), normalised per-frame to [0,1]."""
    T_, H_, W_, C_ = vol.shape
    proj = pca.transform(vol.reshape(-1, C_).numpy()).reshape(T_, H_, W_, 3)
    for t in range(T_):
        lo, hi = proj[t].min(), proj[t].max()
        proj[t] = (proj[t] - lo) / (hi - lo + 1e-8)
    return proj


n_vis = min(VIZ_FRAMES, T_EFF)   # ↑ VIZ_FRAMES from Cell 2

for clip_name, res in all_results.items():
    np.random.seed(0)
    pca = fit_shared_pca(res['bilinear'], res['2d'], res['3d'])

    rgb_bl = vol_to_pca_rgb(res['bilinear'], pca)
    rgb_2d = vol_to_pca_rgb(res['2d'],       pca)
    rgb_3d = vol_to_pca_rgb(res['3d'],       pca)

    rows  = ['Input RGB', 'Bilinear (floor)', 'AnyUp 2D (no temporal ctx)', 'AnyUp 3D (ours)']
    fig, axes = plt.subplots(4, n_vis, figsize=(3.5*n_vis, 4.0*4))

    for col, t in enumerate(range(n_vis)):
        raw_idx = guidance_idx[t]
        axes[0, col].imshow(res['raw'][0, raw_idx].permute(1,2,0).numpy())
        axes[0, col].set_title(f'token {t}\n(raw frame {raw_idx})', fontsize=8)
        axes[1, col].imshow(rgb_bl[t])
        axes[2, col].imshow(rgb_2d[t])
        axes[3, col].imshow(rgb_3d[t])
        for r in range(4): axes[r, col].axis('off')

    for r, label in enumerate(rows):
        axes[r, 0].set_ylabel(label, fontsize=10, rotation=90, labelpad=6)

    m = all_metrics[clip_name]
    plt.suptitle(
        f'{clip_name} — PCA Feature Visualisation (shared basis, VideoMAE features)\n'
        f'Var:   Bilinear={m["bilinear"]["var"]:.4f}  2D={m["2d"]["var"]:.4f}  3D={m["3d"]["var"]:.4f}\n'
        f'Drift: Bilinear={m["bilinear"]["drift"]:.4f}  2D={m["2d"]["drift"]:.4f}  3D={m["3d"]["drift"]:.4f}',
        fontsize=10
    )
    plt.tight_layout(rect=[0, 0, 1, 0.92])
    out_path = f'/content/{clip_name}_pca.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_path}')

## Cell 10 — Frame-diff heatmaps

In [ ]:
# L2 norm of feature difference between consecutive tokens.
# All three rows share the same colour scale per column.

DIFF_PAIRS = min(3, T_EFF - 1)

for clip_name, res in all_results.items():
    fig, axes = plt.subplots(3, DIFF_PAIRS, figsize=(4.5*DIFF_PAIRS, 4.5*3))
    labels = ['Bilinear', 'AnyUp 2D', 'AnyUp 3D']
    vols   = [res['bilinear'], res['2d'], res['3d']]

    for col, t in enumerate(range(DIFF_PAIRS)):
        diffs = [(v[t+1] - v[t]).norm(dim=-1).numpy() for v in vols]
        vmax  = max(d.max() for d in diffs)   # shared scale per column
        for row, (diff, label) in enumerate(zip(diffs, labels)):
            im = axes[row, col].imshow(diff, cmap='hot', vmin=0, vmax=vmax)
            axes[row, col].set_title(f'{label}  |tok{t+1} − tok{t}|', fontsize=9)
            axes[row, col].axis('off')
        plt.colorbar(im, ax=axes[:, col], shrink=0.6, label='L2 change')

    plt.suptitle(f'{clip_name} — Feature-diff heatmaps (shared scale per column)', fontsize=11)
    plt.tight_layout()
    out_path = f'/content/{clip_name}_diff.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_path}')

## Cell 11 — Bar chart (thesis figure, averaged across clips)

In [ ]:
# Average each metric across all evaluated clips for a single summary figure.

methods_order = ['bilinear', '2d', '3d']
method_labels = ['Bilinear\n(floor)', 'AnyUp 2D\n(no temporal)', 'AnyUp 3D\n(ours)']
colors        = ['#aec6cf', '#ff9999', '#77dd77']

avg_var   = {m: np.mean([all_metrics[c][m]['var']   for c in all_metrics]) for m in methods_order}
avg_drift = {m: np.mean([all_metrics[c][m]['drift'] for c in all_metrics]) for m in methods_order}

x = np.arange(len(methods_order))
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

for ax, avg, ylabel, title in [
    (axes[0], avg_var,   'Feature Variance ↓',  'Temporal Feature Variance'),
    (axes[1], avg_drift, 'Mean Cosine Drift ↓', 'Temporal Cosine Drift'),
]:
    vals = [avg[m] for m in methods_order]
    bars = ax.bar(x, vals, color=colors, edgecolor='black', linewidth=0.8)
    ax.set_xticks(x); ax.set_xticklabels(method_labels, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.02,
                f'{val:.5f}', ha='center', va='bottom', fontsize=9)

n_clips = len(all_metrics)
plt.suptitle(
    f'Temporal Consistency Comparison — VideoMAE features\n'
    f'T_eff={T_EFF} tokens, {TARGET_SIZE}×{TARGET_SIZE}  |  averaged over {n_clips} clip(s)\n'
    f'Lower is better for both metrics.',
    fontsize=11
)
plt.tight_layout()
plt.savefig('/content/anyup_metrics_barchart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /content/anyup_metrics_barchart.png')

## Cell 12 — Export side-by-side video (optional)

In [ ]:
import av

def to_uint8(a): return (np.clip(a, 0, 1) * 255).astype(np.uint8)

for clip_name, res in all_results.items():
    np.random.seed(0)
    pca    = fit_shared_pca(res['bilinear'], res['2d'], res['3d'])
    rgb_2d = vol_to_pca_rgb(res['2d'], pca)
    rgb_3d = vol_to_pca_rgb(res['3d'], pca)

    out_path = f'/content/{clip_name}_comparison.mp4'
    FPS = 4   # low FPS makes T_EFF=8 tokens watchable

    with av.open(out_path, mode='w') as container:
        stream = container.add_stream('h264', rate=FPS)
        stream.width   = TARGET_SIZE * 3   # [Input RGB | AnyUp 2D | AnyUp 3D]
        stream.height  = TARGET_SIZE
        stream.pix_fmt = 'yuv420p'

        for t in range(T_EFF):
            raw_idx = guidance_idx[t]
            p0 = to_uint8(res['raw'][0, raw_idx].permute(1,2,0).numpy())
            p1 = to_uint8(rgb_2d[t])
            p2 = to_uint8(rgb_3d[t])
            frame_np = np.concatenate([p0, p1, p2], axis=1)
            for pkt in stream.encode(av.VideoFrame.from_ndarray(frame_np, format='rgb24')):
                container.mux(pkt)
        for pkt in stream.encode(): container.mux(pkt)

    print(f'Saved: {out_path}  [{T_EFF} tokens @ {FPS}fps]')
    print(f'  Layout: [Input RGB | AnyUp 2D PCA | AnyUp 3D PCA]')